In [1]:
!pip install x-transformers pythainlp -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.8/97.8 kB 4.1 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/19.3 MB ? eta -:--:--

   ━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/19.3 MB 132.8 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━ 17.6/19.3 MB 189.1 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 19.3/19.3 MB 187.9 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.0/103.0 kB 9.8 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.7 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import gc

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import f1_score
from torch.amp import autocast, GradScaler

from transformers import AutoTokenizer, AutoModel
from x_transformers.x_transformers import AttentionLayers
from pythainlp.tokenize import word_tokenize
from tqdm import tqdm

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [3]:
# ===== CONFIG =====
TRAIN_CSV = "/kaggle/input/prachatai_train.csv"
VAL_CSV   = "/kaggle/input/prachatai_validation.csv"
TEST_CSV  = "/kaggle/input/prachatai_test.csv"

MODEL_PATH = "mbert_xt_pythai.pt"

MAX_LEN = 256
BATCH_SIZE = 128
EPOCHS = 100
LR = 2e-4

LABEL_COLS = [
    "politics", "human_rights", "quality_of_life", "international",
    "social", "environment", "economics", "culture", "labor",
    "national_security", "ict", "education"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [4]:
hf_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")
PAD_ID = hf_tokenizer.pad_token_id
print("Tokenizer loaded!")

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Tokenizer loaded!


In [5]:
class UltraLazyDataset(Dataset):
    def __init__(self, csv_path):
        self.df = pd.read_csv(csv_path, usecols=["body_text"] + LABEL_COLS)
        self.n = len(self.df)
        print(f"Loaded {self.n} samples from {csv_path}")
        
    def __len__(self):
        return self.n
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        text = str(row["body_text"])
        label = row[LABEL_COLS].values.astype(np.float32)
        
        try:
            words = word_tokenize(text[:1000], engine="newmm")
        except:
            words = text[:1000].split()
            
        enc = hf_tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=MAX_LEN,
            padding=False
        )
        
        return torch.tensor(enc["input_ids"], dtype=torch.long), torch.tensor(label, dtype=torch.float32)

print("Creating datasets...")
train_dataset = UltraLazyDataset(TRAIN_CSV)
val_dataset = UltraLazyDataset(VAL_CSV)
test_dataset = UltraLazyDataset(TEST_CSV)
gc.collect()

Creating datasets...


Loaded 54379 samples from /kaggle/input/prachatai_train.csv


Loaded 6721 samples from /kaggle/input/prachatai_validation.csv


Loaded 6789 samples from /kaggle/input/prachatai_test.csv


34

In [6]:
def collate_fn(batch):
    seqs, labels = zip(*batch)
    padded = pad_sequence(seqs, batch_first=True, padding_value=PAD_ID)
    attn_mask = (padded != PAD_ID).long()
    return padded.to(device), attn_mask.to(device), torch.stack(labels).to(device)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=0)
print(f"DataLoaders ready! Train batches: {len(train_loader)}")

DataLoaders ready! Train batches: 425


In [7]:
encoder = AutoModel.from_pretrained("bert-base-multilingual-cased")
for p in encoder.parameters():
    p.requires_grad = False
print("Encoder loaded (frozen)")

2026-02-08 07:16:21.932165: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770534982.111892      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770534982.161412      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770534982.580687      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770534982.580715      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770534982.580718      24 computation_placer.cc:177] computation placer alr

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Encoder loaded (frozen)


In [8]:
class MBertXTClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = encoder
        self.hidden = 768

        self.decoder = AttentionLayers(
            dim=self.hidden,
            depth=4,
            heads=4,
            cross_attend=True,
            causal=False
        )
        self.classifier = nn.Linear(self.hidden, len(LABEL_COLS))

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            enc = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        
        x = torch.zeros_like(enc)
        dec = self.decoder(x, context=enc, context_mask=attention_mask.bool())
        
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (dec * mask).sum(1) / mask.sum(1).clamp(min=1)
        return self.classifier(pooled)

model = MBertXTClassifier().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
scaler = GradScaler()

print(f"Model ready! Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} params")

Model ready! Trainable: 25,200,396 params


In [9]:
# ===== TRAINING (Save every epoch) =====
print("Starting training...")
best_f1 = 0.0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}")
    for i, (Xb, maskb, yb) in enumerate(pbar):
        optimizer.zero_grad()
        
        with autocast(device_type="cuda"):
            logits = model(Xb, maskb)
            loss = criterion(logits, yb)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        
        if (i + 1) % 50 == 0:
            torch.cuda.empty_cache()
            pbar.set_postfix({"loss": f"{total_loss/(i+1):.4f}"})

    # Validation
    model.eval()
    yt, yp = [], []
    with torch.no_grad():
        for Xb, maskb, yb in val_loader:
            with autocast(device_type="cuda"):
                logits = model(Xb, maskb)
                preds = (torch.sigmoid(logits) > 0.5).int()
            yt.append(yb.cpu().numpy())
            yp.append(preds.cpu().numpy())

    val_f1 = f1_score(np.vstack(yt), np.vstack(yp), average="macro")
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1:03d} | Loss {avg_loss:.4f} | Val F1 {val_f1:.4f}")

    # 💾 Save ทุก epoch (เก็บ best model ไว้)
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), MODEL_PATH)
        print(f"  💾 Saved best model (F1={val_f1:.4f})")
    
    torch.cuda.empty_cache()
    gc.collect()

print(f"\nTraining complete! Best Val F1: {best_f1:.4f}")

Starting training...


Epoch 1:   0%|          | 0/425 [00:00<?, ?it/s]

Epoch 1:   0%|          | 1/425 [00:02<20:30,  2.90s/it]

Epoch 1:   0%|          | 2/425 [00:05<17:51,  2.53s/it]

Epoch 1:   1%|          | 3/425 [00:07<17:00,  2.42s/it]

...

Epoch 40:  94%|█████████▍| 401/425 [15:35<00:57,  2.38s/it, loss=0.0440]

Epoch 40:  95%|█████████▍| 402/425 [15:37<00:54,  2.36s/it, loss=0.0440]

Epoch 40:  95%|█████████▍| 403/425 [15:39<00:52,  2.36s/it, loss=0.0440]

Epoch 40:  95%|█████████▌| 404/425 [15:42<00:49,  2.35s/it, loss=0.0440]

Epoch 40:  95%|█████████▌| 405/425 [15:44<00:46,  2.34s/it, loss=0.0440]

In [ ]:
# ===== TEST EVALUATION =====
import os
if os.path.exists(MODEL_PATH):
    model.load_state_dict(torch.load(MODEL_PATH))
    print("Loaded best model!")
else:
    print("Using current model (no checkpoint found)")

model.eval()

yt, yp = [], []
with torch.no_grad():
    for Xb, maskb, yb in tqdm(test_loader, desc="Testing"):
        with autocast(device_type="cuda"):
            logits = model(Xb, maskb)
            preds = (torch.sigmoid(logits) > 0.5).int()
        yt.append(yb.cpu().numpy())
        yp.append(preds.cpu().numpy())

test_f1 = f1_score(np.vstack(yt), np.vstack(yp), average="macro")
print(f"\n🎯 Test F1 Score: {test_f1:.4f}")

In [ ]:
# ===== PREDICTION =====
def predict(text, threshold=0.5):
    words = word_tokenize(text[:500], engine="newmm")
    enc = hf_tokenizer(words, is_split_into_words=True, return_tensors="pt", truncation=True, max_length=MAX_LEN)
    
    with torch.no_grad():
        with autocast(device_type="cuda"):
            logits = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))
            probs = torch.sigmoid(logits)[0].cpu().numpy()
    
    results = [(LABEL_COLS[i], float(probs[i])) for i in range(len(LABEL_COLS)) if probs[i] >= threshold]
    return sorted(results, key=lambda x: x[1], reverse=True)

print("\n===== PREDICT EXAMPLES =====")
print(predict("รัฐบาลไทยประกาศนโยบายด้านสิ่งแวดล้อมใหม่"))
print(predict("แรงงานเรียกร้องสิทธิ์การทำงาน"))